# 02 — Exploratory Data Analysis
## Nyando Flood AI · James Koero

**Goal:** Understand the real GEE training data before modelling.
I want to check distributions, correlations, class balance, and whether features
make physical sense for a flood basin in western Kenya.

**Expected findings:**
- Low elevation + high rainfall + close river distance → flood
- Clay-rich soils reduce infiltration → flood more likely
- Water/wetland land cover classes → higher flood rate

**If I find something unexpected, I will document it as an error and investigate.**

In [ ]:
!pip install matplotlib pandas numpy -q
import pandas as pd, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

NAVY='#0A1628'; GOLD='#C9A84C'; TEAL='#2EC4B6'; RED='#E63946'; LGRAY='#8A9BB0'

# Load data
df = pd.read_csv('https://raw.githubusercontent.com/jameskoero/nyando-flood-ai/main/data/training/nyando_training_v1.csv')
FEATURES = ['elevation','slope','rainfall_3day','distance_river','clay_percent','land_cover']
print(f'Shape: {df.shape}')
print(f'Flood rate: {df["flooded"].mean():.1%}')
print(f'Nulls: {df.isnull().sum().sum()}')
df.describe().round(2)

### Class Balance Check
I want to find out how imbalanced the classes are, because this will determine whether I need SMOTE.
If flood rate < 15% or > 50%, SMOTE will be necessary.

In [ ]:
flood_rate = df['flooded'].mean()
print(f'Not flooded: {(df.flooded==0).sum():,} ({1-flood_rate:.1%})')
print(f'Flooded:     {(df.flooded==1).sum():,} ({flood_rate:.1%})')
print(f'Imbalance ratio: {(df.flooded==0).sum()/(df.flooded==1).sum():.1f}:1')
print()
if flood_rate < 0.35:
    print('⚠️  Class imbalance confirmed — SMOTE required in notebook 03')
else:
    print('✅ Classes reasonably balanced')

### Feature Distributions
Plotting each feature split by flood/no-flood to check for discriminating power.

In [ ]:
fig, axes = plt.subplots(2,3,figsize=(14,8)); fig.patch.set_facecolor(NAVY)
titles = ['Elevation (m)','Slope (°)','Rainfall 3-day (mm)',
          'River Distance (m)','Clay % 0-5cm','Land Cover Class']
for ax, feat, title in zip(axes.flat, FEATURES, titles):
    ax.set_facecolor('#0E1E35')
    for sp in ax.spines.values(): sp.set_color(GOLD)
    ax.tick_params(colors=LGRAY)
    df[df.flooded==0][feat].hist(ax=ax,bins=30,alpha=.65,color=TEAL,label='Not Flooded')
    df[df.flooded==1][feat].hist(ax=ax,bins=30,alpha=.65,color=RED, label='Flooded')
    ax.set_title(title,color=GOLD,fontsize=9,fontweight='bold')
    ax.legend(fontsize=7,facecolor='#0E1E35',edgecolor=GOLD,labelcolor='white')
    ax.grid(alpha=.1,color=LGRAY)
fig.suptitle(f'Feature Distributions — {len(df):,} Real GEE Points | Nyando Basin',
             color=GOLD,fontsize=12,fontweight='bold',y=1.01)
plt.tight_layout()
plt.savefig('reports/figures/eda_distributions.png',dpi=150,bbox_inches='tight',facecolor=NAVY)
plt.show()
print('EDA distributions saved ✅')

### Correlation Analysis
I want to understand how features relate to each other and to the flood label.
High correlation between features could cause redundancy.

In [ ]:
corr = df[FEATURES + ['flooded']].corr()
fig, ax = plt.subplots(figsize=(9,7)); fig.patch.set_facecolor(NAVY)
ax.set_facecolor('#0E1E35')
im = ax.imshow(corr, cmap='RdYlBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
cols = FEATURES + ['flooded']
ax.set_xticks(range(len(cols))); ax.set_yticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=45, ha='right', color='white', fontsize=8)
ax.set_yticklabels(cols, color='white', fontsize=8)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j,i,f'{corr.values[i,j]:.2f}',ha='center',va='center',
                color='white' if abs(corr.values[i,j])<0.5 else NAVY,fontsize=7)
ax.set_title('Feature Correlation Matrix',color=GOLD,fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation with flood label
print('\nCorrelation with flood label:')
flood_corr = corr['flooded'].drop('flooded').sort_values(key=abs, ascending=False)
for feat, val in flood_corr.items():
    direction = '⬆️' if val > 0 else '⬇️'
    print(f'  {feat:20s}: {val:+.3f} {direction}')

### Land Cover Analysis
Checking which land cover classes show highest flood rates.

In [ ]:
# ESA WorldCover class labels
lc_labels = {10:'Tree cover',20:'Shrubland',30:'Grassland',40:'Cropland',
             50:'Built-up',60:'Bare/sparse',70:'Snow/ice',80:'Water bodies',
             90:'Wetlands',95:'Mangroves',100:'Moss/lichen'}

lc_flood = df.groupby('land_cover').agg(
    count=('flooded','count'), flood_rate=('flooded','mean')
).reset_index()
lc_flood['label'] = lc_flood['land_cover'].map(lc_labels).fillna('Other')
lc_flood = lc_flood.sort_values('flood_rate', ascending=False)

print('Land cover flood rates:')
print(lc_flood[['label','count','flood_rate']].to_string(index=False))
print()
print('✅ Expected: Water(80) and Wetlands(90) should have highest flood rates')

### Geographic Distribution (lon/lat)
Checking that sample points actually cover the Nyando Basin bounds.

In [ ]:
if 'lon' in df.columns and 'lat' in df.columns:
    fig, ax = plt.subplots(figsize=(9,6)); fig.patch.set_facecolor(NAVY)
    ax.set_facecolor('#0E1E35')
    ax.scatter(df[df.flooded==0]['lon'], df[df.flooded==0]['lat'],
               c=TEAL, s=3, alpha=.4, label='Not Flooded')
    ax.scatter(df[df.flooded==1]['lon'], df[df.flooded==1]['lat'],
               c=RED, s=5, alpha=.7, label='Flooded')
    ax.set_xlabel('Longitude', color=LGRAY)
    ax.set_ylabel('Latitude', color=LGRAY)
    ax.set_title(f'Spatial Distribution — Nyando Basin | {len(df):,} Points',
                 color=GOLD, fontweight='bold')
    ax.legend(facecolor='#0E1E35',edgecolor=GOLD,labelcolor='white')
    for sp in ax.spines.values(): sp.set_color(GOLD)
    ax.tick_params(colors=LGRAY)
    plt.tight_layout()
    plt.show()
else:
    print('No lon/lat columns in this dataset version')

print('\n=== EDA Summary ===')
print(f'Total points: {len(df):,}')
print(f'Flood rate: {df["flooded"].mean():.1%}')
print(f'Elevation range: {df.elevation.min():.0f}–{df.elevation.max():.0f}m')
print(f'Rainfall range: {df.rainfall_3day.min():.1f}–{df.rainfall_3day.max():.1f}mm')
print('\n→ Proceed to notebook 03_modelling.ipynb')